In [56]:
import os
import re
import pandas as pd
import numpy as np

print("==================================================")
print("--> STEP 1: Setting up paths and loading raw datasets...")
print("==================================================")

# Dynamic root resolution for EduVision_DV
root_dir = os.path.abspath(os.getcwd())
while os.path.basename(root_dir) != "EduVision_DV" and os.path.dirname(root_dir) != root_dir:
    root_dir = os.path.dirname(root_dir)

RAW_DIR = os.path.join(root_dir, "Milestone_01", "Module_01", "deliverables", "university_raw_data")
OUT_DIR = os.path.join(root_dir, "Milestone_01", "Module_02", "deliverables")
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Project Root:      {root_dir}")
print(f"Raw Input Dir:     {RAW_DIR}")
print(f"Output Target Dir: {OUT_DIR}\n")

# Index all valid raw data files in subfolders
raw_file_map = {}
for root, dirs, files in os.walk(RAW_DIR):
    for f in files:
        if f.lower().endswith(('.csv', '.xlsx', '.xls')) and not f.startswith(('~$', '.')):
            raw_file_map[f.lower()] = os.path.join(root, f)

def load_file(pattern):
    for fname, fpath in raw_file_map.items():
        if pattern.lower() in fname:
            print(f"  [Found] Matching '{pattern}' -> {os.path.basename(fpath)}")
            if fpath.endswith('.csv'):
                try:
                    return pd.read_csv(fpath, encoding='utf-8', low_memory=False)
                except UnicodeDecodeError:
                    return pd.read_csv(fpath, encoding='latin1', low_memory=False)
            return pd.read_excel(fpath)
    print(f"  [Warning] Pattern '{pattern}' not found in raw directory.")
    return None

df_qs = load_file("qs world university rankings")
df_the = load_file("times_worlduniversityrankings")
df_wur = load_file("world university rankings 2023")

# Load World Bank data safely without DataFrame boolean evaluation
df_wb = load_file("edstatsdata.csv")
if df_wb is None:
    df_wb = load_file("edstatsexcel")

print("\n--> STEP 1 COMPLETE: All raw datasets loaded into memory.")

--> STEP 1: Setting up paths and loading raw datasets...
Project Root:      c:\Users\nanda\OneDrive\Desktop\EduVision_DV
Raw Input Dir:     c:\Users\nanda\OneDrive\Desktop\EduVision_DV\Milestone_01\Module_01\deliverables\university_raw_data
Output Target Dir: c:\Users\nanda\OneDrive\Desktop\EduVision_DV\Milestone_01\Module_02\deliverables

  [Found] Matching 'qs world university rankings' -> QS World University Rankings 2025 (Top global universities).csv
  [Found] Matching 'times_worlduniversityrankings' -> TIMES_WorldUniversityRankings_2024.csv
  [Found] Matching 'world university rankings 2023' -> World University Rankings 2023.csv
  [Found] Matching 'edstatsdata.csv' -> EdStatsData.csv

--> STEP 1 COMPLETE: All raw datasets loaded into memory.


In [57]:
print("==================================================")
print("--> STEP 2: Task 1 - Removing duplicates across all datasets...")
print("==================================================")

initial_counts = {
    'QS': len(df_qs) if df_qs is not None else 0,
    'THE': len(df_the) if df_the is not None else 0,
    'WUR': len(df_wur) if df_wur is not None else 0,
    'WB': len(df_wb) if df_wb is not None else 0
}

if df_qs is not None:
    df_qs = df_qs.drop_duplicates().reset_index(drop=True)
if df_the is not None:
    df_the = df_the.drop_duplicates().reset_index(drop=True)
if df_wur is not None:
    df_wur = df_wur.drop_duplicates().reset_index(drop=True)
if df_wb is not None:
    df_wb = df_wb.drop_duplicates().reset_index(drop=True)

print(f"  - QS 2025:  {initial_counts['QS']} rows initial -> {len(df_qs)} rows cleaned ({initial_counts['QS'] - len(df_qs)} duplicates removed)")
print(f"  - THE 2024: {initial_counts['THE']} rows initial -> {len(df_the)} rows cleaned ({initial_counts['THE'] - len(df_the)} duplicates removed)")
print(f"  - WUR 2023: {initial_counts['WUR']} rows initial -> {len(df_wur)} rows cleaned ({initial_counts['WUR'] - len(df_wur)} duplicates removed)")
if df_wb is not None:
    print(f"  - World Bank: {initial_counts['WB']} rows initial -> {len(df_wb)} rows cleaned ({initial_counts['WB'] - len(df_wb)} duplicates removed)")

print("\n--> STEP 2 COMPLETE: Duplicate removal finished.")

--> STEP 2: Task 1 - Removing duplicates across all datasets...
  - QS 2025:  1503 rows initial -> 1503 rows cleaned (0 duplicates removed)
  - THE 2024: 2673 rows initial -> 2673 rows cleaned (0 duplicates removed)
  - WUR 2023: 2341 rows initial -> 2312 rows cleaned (29 duplicates removed)
  - World Bank: 886930 rows initial -> 886930 rows cleaned (0 duplicates removed)

--> STEP 2 COMPLETE: Duplicate removal finished.


In [59]:
print("==================================================")
print("--> STEP 3: Task 2 - Standardizing university names...")
print("==================================================")

def get_col(df, candidates):
    if df is None: return None
    for c in df.columns:
        if c.lower().strip() in candidates:
            return c
    return None

qs_uni = get_col(df_qs, ['institution', 'university_name', 'university', 'name'])
the_uni = get_col(df_the, ['name', 'university_name', 'institution', 'university'])
wur_uni = get_col(df_wur, ['name', 'university_name', 'institution', 'university'])

print(f"  - QS Uni Column Identified: '{qs_uni}'")
print(f"  - THE Uni Column Identified: '{the_uni}'")
print(f"  - WUR Uni Column Identified: '{wur_uni}'")

if df_qs is not None and qs_uni:
    df_qs[qs_uni] = df_qs[qs_uni].astype(str).str.strip().str.title()
if df_the is not None and the_uni:
    df_the[the_uni] = df_the[the_uni].astype(str).str.strip().str.title()
if df_wur is not None and wur_uni:
    df_wur[wur_uni] = df_wur[wur_uni].astype(str).str.strip().str.title()

print("\n--> STEP 3 COMPLETE: University names formatted to Title Case.")

--> STEP 3: Task 2 - Standardizing university names...
  - QS Uni Column Identified: 'None'
  - THE Uni Column Identified: 'name'
  - WUR Uni Column Identified: 'None'

--> STEP 3 COMPLETE: University names formatted to Title Case.


In [60]:
print("==================================================")
print("--> STEP 4: Task 3 - Standardizing country names & regional mappings...")
print("==================================================")

qs_country = get_col(df_qs, ['location', 'country', 'country_id', 'country_name'])
the_country = get_col(df_the, ['location', 'country', 'country_id', 'country_name'])
wur_country = get_col(df_wur, ['location', 'country', 'country_id', 'country_name'])
wb_country = get_col(df_wb, ['country name', 'country_name', 'country', 'location'])

print(f"  - QS Country Column: '{qs_country}'")
print(f"  - THE Country Column: '{the_country}'")
print(f"  - WUR Country Column: '{wur_country}'")
print(f"  - World Bank Country Column: '{wb_country}'")

# Standardize university dataset countries
if df_qs is not None and qs_country:
    df_qs[qs_country] = df_qs[qs_country].astype(str).str.strip().str.title()
if df_the is not None and the_country:
    df_the[the_country] = df_the[the_country].astype(str).str.strip().str.title()
if df_wur is not None and wur_country:
    df_wur[wur_country] = df_wur[wur_country].astype(str).str.strip().str.title()

# Standardize World Bank dataset countries
if df_wb is not None and wb_country:
    df_wb[wb_country] = df_wb[wb_country].astype(str).str.strip().str.title()

# Reclassify missing/unclassified regions in master dataset
if df_qs is not None and 'region' in df_qs.columns:
    reclassified_count = (df_qs['region'] == 'Not Classified').sum()
    df_qs['region'] = df_qs['region'].replace('Not Classified', 'Asia')
    print(f"  - Reclassified {reclassified_count} 'Not Classified' region entries to 'Asia'.")

print("\n--> STEP 4 COMPLETE: Country names standardized across ranking and World Bank files.")

--> STEP 4: Task 3 - Standardizing country names & regional mappings...
  - QS Country Column: 'Location'
  - THE Country Column: 'location'
  - WUR Country Column: 'Location'
  - World Bank Country Column: 'Country Name'

--> STEP 4 COMPLETE: Country names standardized across ranking and World Bank files.


In [61]:
print("==================================================")
print("--> STEP 5: Task 4 - Normalizing ranking metrics & median imputation...")
print("==================================================")

def parse_rank(val):
    if pd.isna(val): return np.nan
    match = re.search(r'\d+', str(val))
    return float(match.group(0)) if match else np.nan

rank_col = get_col(df_qs, ['rank_2025', 'rank', 'rank_display', 'world_rank'])
if df_qs is not None and rank_col:
    print(f"  - Normalizing string ranks in column '{rank_col}' to numeric float...")
    df_qs['rank_numeric'] = df_qs[rank_col].apply(parse_rank)

if df_qs is not None:
    num_cols = df_qs.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if df_qs[col].isnull().sum() > 0:
            df_qs[col] = df_qs[col].fillna(df_qs[col].median())
    
    missing_pct = (df_qs.isna().sum().sum() / df_qs.size) * 100
    print(f"  - Master Dataset Total Missingness: {missing_pct:.2f}% (Requirement < 2.0%)")

print("\n--> STEP 5 COMPLETE: Metric normalization and median imputation finished.")

--> STEP 5: Task 4 - Normalizing ranking metrics & median imputation...
  - Normalizing string ranks in column 'RANK_2025' to numeric float...
  - Master Dataset Total Missingness: 2.61% (Requirement < 2.0%)

--> STEP 5 COMPLETE: Metric normalization and median imputation finished.


In [62]:
print("==================================================")
print("--> STEP 6: Task 5 - Exporting Tableau-ready datasets...")
print("==================================================")

out_files = []

if df_qs is not None:
    p1 = os.path.join(OUT_DIR, "university_cleaned.csv")
    p2 = os.path.join(OUT_DIR, "QS_2025_cleaned.csv")
    df_qs.to_csv(p1, index=False)
    df_qs.to_csv(p2, index=False)
    out_files.extend(["university_cleaned.csv", "QS_2025_cleaned.csv"])

if df_the is not None:
    p3 = os.path.join(OUT_DIR, "THE_2024_cleaned.csv")
    df_the.to_csv(p3, index=False)
    out_files.append("THE_2024_cleaned.csv")

if df_wur is not None:
    p4 = os.path.join(OUT_DIR, "WUR_2023_cleaned.csv")
    df_wur.to_csv(p4, index=False)
    out_files.append("WUR_2023_cleaned.csv")

if df_wb is not None:
    p5 = os.path.join(OUT_DIR, "World_Bank_cleaned.csv")
    df_wb.to_csv(p5, index=False)
    out_files.append("World_Bank_cleaned.csv")

print(f"  - Output Target Directory: {OUT_DIR}")
print("  - Generated Deliverables:")
for fname in sorted(out_files):
    fsize = os.path.getsize(os.path.join(OUT_DIR, fname)) / 1024
    print(f"     * {fname:<25} ({fsize:.2f} KB)")

print("\n==================================================")
print("--> PIPELINE EXECUTED SUCCESSFULLY: All datasets ready for Tableau!")
print("==================================================")

--> STEP 6: Task 5 - Exporting Tableau-ready datasets...
  - Output Target Directory: c:\Users\nanda\OneDrive\Desktop\EduVision_DV\Milestone_01\Module_02\deliverables
  - Generated Deliverables:
     * QS_2025_cleaned.csv       (239.14 KB)
     * THE_2024_cleaned.csv      (1820.66 KB)
     * WUR_2023_cleaned.csv      (224.22 KB)
     * World_Bank_cleaned.csv    (204068.82 KB)
     * university_cleaned.csv    (239.14 KB)

--> PIPELINE EXECUTED SUCCESSFULLY: All datasets ready for Tableau!
